# Atelier Fine-Tuning — Google Colab

**What this notebook does:**  
1. Installs QLoRA dependencies (bitsandbytes, trl, peft)  
2. Mounts your Drive to load the dataset  
3. Runs QLoRA fine-tuning on Mistral-7B-v0.3  
4. Merges adapter + exports GGUF  

**Requirements:**  
- Colab Pro+ with A100 GPU (recommended) or at minimum L4/T4 (slower)  
- HuggingFace token with write access  
- Upload `train.jsonl` and `val.jsonl` to your Drive at `MyDrive/atelier/dataset/`

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout)
print('GPU check done. Proceed if you see A100/H100/L4.')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
!pip install -q \
  'transformers>=4.44.0' \
  'datasets>=2.20.0' \
  'trl>=0.9.6' \
  'peft>=0.12.0' \
  'bitsandbytes>=0.43.0' \
  'accelerate>=0.33.0' \
  'sentencepiece' \
  'safetensors'
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Mount Drive & set up paths ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
from pathlib import Path

# Dataset paths (edit if you put them somewhere else in Drive)
DRIVE_DATASET = Path('/content/drive/MyDrive/atelier/dataset')
LOCAL_DATASET = Path('/content/atelier/model/dataset')
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

shutil.copy(DRIVE_DATASET / 'train.jsonl', LOCAL_DATASET / 'train.jsonl')
shutil.copy(DRIVE_DATASET / 'val.jsonl',   LOCAL_DATASET / 'val.jsonl')

!wc -l /content/atelier/model/dataset/train.jsonl /content/atelier/model/dataset/val.jsonl
print('Dataset ready.')

In [ ]:
# ── Cell 4: Clone repo & configure paths ────────────────────────────────────
!git clone https://github.com/anshrajore/atelier-mcp.git /content/atelier-repo

# Copy train scripts into local workspace
import shutil
for f in ['train.py', 'merge.py', 'export_gguf.py', 'config.yaml']:
    shutil.copy(f'/content/atelier-repo/model/train/{f}', f'/content/atelier/model/train/{f}')

print('Scripts ready.')

In [ ]:
# ── Cell 5: HuggingFace login ────────────────────────────────────────────────
from huggingface_hub import login
from google.colab import userdata

# Store your HF token as a Colab Secret named HF_TOKEN
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('HuggingFace login successful.')

In [ ]:
# ── Cell 6: Patch config.yaml for Colab paths ───────────────────────────────
import yaml
from pathlib import Path

config_path = Path('/content/atelier/model/train/config.yaml')
config = yaml.safe_load(config_path.read_text())

config['train_dataset'] = '/content/atelier/model/dataset/train.jsonl'
config['val_dataset']   = '/content/atelier/model/dataset/val.jsonl'
config['output_dir']    = '/content/atelier/model/output/atelier-7b-lora'

# A100: can use 4-bit NF4. T4: keep 4-bit but reduce batch_size to 2.
config['per_device_train_batch_size'] = 4   # reduce to 2 for T4
config['gradient_accumulation_steps'] = 8
config['num_train_epochs'] = 3

config_path.write_text(yaml.dump(config))
print('Config patched:')
print(yaml.dump(config))

In [ ]:
# ── Cell 7: Run QLoRA Training ───────────────────────────────────────────────
# Estimated time: ~2-3h on A100, ~6-8h on T4
%cd /content/atelier
!python3 model/train/train.py

In [ ]:
# ── Cell 8: Merge adapter into full model ────────────────────────────────────
!python3 model/train/merge.py
print('Merged model at /content/atelier/model/output/atelier-7b-merged/')

In [ ]:
# ── Cell 9: Export GGUF (optional) ──────────────────────────────────────────
!python3 model/train/export_gguf.py
print('GGUF exported.')

In [ ]:
# ── Cell 10: Save outputs to Drive ───────────────────────────────────────────
import shutil
from pathlib import Path

DRIVE_OUT = Path('/content/drive/MyDrive/atelier/output')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

# Copy merged model
shutil.copytree(
    '/content/atelier/model/output/atelier-7b-merged',
    str(DRIVE_OUT / 'atelier-7b-merged'),
    dirs_exist_ok=True
)

# Copy GGUF if exists
gguf = Path('/content/atelier/model/output/atelier-7b-q4.gguf')
if gguf.exists():
    shutil.copy(gguf, DRIVE_OUT / 'atelier-7b-q4.gguf')

print(f'Saved to {DRIVE_OUT}')

In [ ]:
# ── Cell 11: (Optional) Push to HuggingFace Hub ─────────────────────────────
from huggingface_hub import HfApi

api = HfApi()
# Change to your HF username
HF_USERNAME = 'anshrajore'
REPO_ID = f'{HF_USERNAME}/atelier-7b'

api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(
    folder_path='/content/atelier/model/output/atelier-7b-merged',
    repo_id=REPO_ID,
    repo_type='model'
)
print(f'Pushed to https://huggingface.co/{REPO_ID}')